In [0]:
%run /Users/manoranjanpandey836@gmail.com/Fake_API/config_utility

In [0]:
trans_df = spark.table("workspace.bronze.finance_transactions_bronze")
trans_df.show()

### Null checks

In [0]:
# trans_df.count() # - Record count check
from pyspark.sql.functions import col, sum
trans_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in trans_df.columns
]
).show()

### Duplicate check on transaction_id

In [0]:
from pyspark.sql.functions import count


dups_count = trans_df.groupBy("transaction_id") \
    .agg(count("*").alias("dups_row")) \
    .filter("dups_row > 1") \
    .count()
    
print("Duplicate transaction id",dups_count) 





### Extract Year, month from transaction_date column 

In [0]:
import pyspark.sql.functions as F
trans_df = trans_df.withColumn(
    "transaction_year", F.year("transaction_date")) \
    .withColumn("transaction_month", F.date_format(F.col("transaction_date"),"MMM")) \
    .withColumn("Loaded_date", F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm"))


In [0]:
display(trans_df)

In [0]:
def write_to_table():
    trans_df.write \
        .format("delta") \
        .mode(silver["write_mode"]) \
        .saveAsTable(silver["full_table_name"])

    return True


df_wirte = write_to_table()

print("Silver table created successfully!")
print("Table name:", silver["full_table_name"])